## Lesson 1: The "Profit Leak" Detection (Thinking Process & SQL)
Most analysts are "order takers"—they pull what they are asked for. A 100x analyst finds what the business didn't know was missing.

#### 1. The Scenario
Our company’s revenue is up, but the bank account isn't growing at the same rate. Management thinks it’s "marketing costs." You suspect it’s transactional slippage (failed payments, unapplied discounts, or ghost refunds).

#### 2. The Thinking Process (Root Cause Analysis)
Before coding, we use the MECE Framework (Mutually Exclusive, Collectively Exhaustive) to categorize where money could be "leaking":
- Inbound: Are we charging but not collecting? (Gateway failures)
- Outbound: Are we over-refunding? (System glitches)
- Internal: Are discounts being stacked incorrectly? (Logic errors)

#### 3. The Execution (SQL Strategy)
Don't just join Orders and Payments. We need to find the orphans.
- The 100x approach: Finding 'Ghost Revenue'
- We look for orders that have a 'Success' status but no matching 
- confirmed transaction in the ledger within a 2-hour window.

In [ ]:
WITH Order_Flow AS (
    SELECT 
        o.order_id,
        o.order_amount,
        o.created_at,
        p.transaction_id,
        p.status AS payment_status
    FROM orders o
    LEFT JOIN payments p 
        ON o.order_id = p.order_id
        AND p.created_at BETWEEN o.created_at AND DATE_ADD(o.created_at, INTERVAL 2 HOUR)
)
SELECT 
    DATE(created_at) AS gap_date,
    COUNT(order_id) AS missing_payment_count,
    SUM(order_amount) AS potential_leakage
FROM Order_Flow
WHERE transaction_id IS NULL
GROUP BY 1
HAVING potential_leakage > 0
ORDER BY potential_leakage DESC;

#### 4. The Solution Provisioning & Communication
- The Junior approach: "I ran a query and found 50 orders without payments."
- The 100x approach: "I’ve identified a sync lag between our checkout and the payment gateway. We processed $12,400 in 'successful' orders yesterday that have no corresponding bank record. I’ve flagged the specific Order IDs for the engineering team and set up a Slack alert to catch this in real-time."

#### Your 10-Minute Drill
To wrap up today’s 30-minute block, give me your thoughts on this:

If you found that the "leakage" was actually caused by a specific discount code being used twice on the same order, how would you change your SQL approach to catch "Multi-Discount Abuse"?

Write out the logic (or the code) and I’ll critique it. Tomorrow, we move into Python Automation for these types of alerts. Would you like to start with the SQL logic for that discount abuse check?

#### 1x Analyst Response
If it were to be a specific discount code used twice on same order, I would check that pulls the order against the discount code, using having to check if a specific order id appears multiple times. This will also catch orders that have multiple discount codes used (which should ideally not be so)

#### 100x Analyst Response
Spot on. You’ve hit the core of Set Logic. By grouping by the order_id and the discount_code, you are isolating the "grain" of the abuse.

The 100x Critique:

Your logic is structurally sound, but a 100x analyst looks for the Exploit Pattern. If a user found a way to apply the same code twice, they might have also found a way to "stack" different codes (e.g., a "Welcome10" plus a "Spring20").

Here is how we translate your logic into a Hardened SQL Query that catches both:

In [ ]:
SELECT 
    order_id,
    user_id,
    COUNT(discount_code) AS codes_applied,
    COUNT(DISTINCT discount_code) AS unique_codes_applied,
    SUM(discount_amount) AS total_discount_hit
FROM order_discounts
GROUP BY 1, 2
HAVING 
    -- Logic 1: Catching the same code used twice 
    -- (e.g., Code 'SAVE10' used 2x on 1 order)
    COUNT(discount_code) > COUNT(DISTINCT discount_code)

    OR 

    -- Logic 2: Catching "Stacking" 
    -- (e.g., Code 'SAVE10' + 'WELCOME5' used on 1 order)
    COUNT(DISTINCT discount_code) > 1

    OR

    -- Original Business Rule: Catching excessive total value
    SUM(discount_amount) > (SELECT MAX_ALLOWED_DISCOUNT FROM business_rules)

ORDER BY total_discount_hit DESC;

#### Breakdown for more refined query below - Identifying Margin Erosion from Discount Abuse
The Logic: You correctly identified that COUNT(discount_code) > 1 is the "smoke." My addition of the discount_intensity (ratio) is the "fire." If a user applies two $5 coupons on a $1,000 order, it's a bug but not a crisis. If they apply two $5 coupons on a $12 order, they are eating our shipping costs and profit.

The Business Impact: By joining back to the orders table, we can tell the CEO: "We aren't just seeing double-discounts; we are seeing 400 orders where the net revenue was less than the cost of goods sold."

In [ ]:
WITH Discount_Abuse_List AS (
    SELECT 
        order_id,
        user_id,
        COUNT(discount_code) AS codes_applied,
        COUNT(DISTINCT discount_code) AS unique_codes_applied,
        SUM(discount_amount) AS total_discount_hit
    FROM order_discounts
    GROUP BY 1, 2
    -- Logic: More than one code OR exceeding a hard business threshold
    HAVING COUNT(discount_code) > 1 
       OR SUM(discount_amount) > 50.00 -- Replacing placeholder with a concrete logic threshold
)

SELECT 
    da.order_id,
    da.user_id,
    da.codes_applied,
    da.total_discount_hit,
    o.order_amount AS gross_revenue,
    (o.order_amount - da.total_discount_hit) AS net_revenue,
    -- Calculate the 'Abuse Ratio' to prioritize the worst offenders
    (da.total_discount_hit / NULLIF(o.order_amount, 0)) AS discount_intensity
FROM Discount_Abuse_List da
JOIN orders o ON da.order_id = o.order_id
WHERE (o.order_amount - da.total_discount_hit) < (o.order_amount * 0.5) -- Flagging if margin drops below 50%
ORDER BY discount_intensity DESC;